In [12]:
import cassiopeia as cas
import pandas as pd
import os
import glob
from ete3 import Tree
from cassiopeia.data import CassiopeiaTree
from io import StringIO
from Bio import Phylo
import numpy as np

# 0. load state file

In [47]:
sc = pd.read_csv("/syn2/zhaolian/3.JiLab/results/2.scRNAseq/4.integrated_byTypeReClustering/step5.scdata_metadata_nod_1_20_0.2.txt", sep="\t")
sc = sc.rename(columns={"cellID": "cell"})
sc['organ'] = sc['sampleIDnew'].str.split("_").str[1]

metgroup = pd.read_csv("/syn2/zhaolian/3.JiLab/results/2.scRNAseq/4.integrated_byTypeReClustering/step8.nod_row_annotation_data0.98.txt", sep="\t")
metgroup['mouse'] = metgroup.index.str.split("_").str[0]
metgroup['mouse'] = "C" + metgroup['mouse']
metgroup['clone'] = metgroup.index.str.split("_").str[1]


# 1. prune tree

In [ ]:
out_dir="/syn2/zhaolian/3.JiLab/results/3.PacBio/6.fitchCount_5cells"
if not os.path.exists(out_dir):
    os.makedirs(out_dir)
os.chdir(out_dir)

for my_clone in range(len(metgroup)):
    mouse = metgroup.iloc[my_clone]['mouse']
    clone = metgroup.iloc[my_clone]['clone']
    
    input_dir = f"/syn2/zhaolian/3.JiLab/results/3.PacBio/4.clones_filtered/{mouse}"
    files = [f for f in os.listdir(input_dir) if f.startswith(f"Clone{clone}_") and f.endswith("_rescale.nwk")]
    
    if len(files) != 1:
        # print(f"Skipping clone {my_clone} due to ambiguous {len(files)} files.")
        continue
    
    treefile = os.path.join(input_dir, files[0])
    outname = f"{metgroup.index[my_clone]}_pruned.nwk"
    outfile = os.path.join(out_dir, outname)
    
    # 读取树
    with open(treefile, 'r') as f:
        tree_str = f.read()
    
    handle = StringIO(tree_str)
    tree = Phylo.read(handle, "newick")
    
    terminals = tree.get_terminals()
    ######################################################
    # 第一步筛选：去掉ref和不在sc['cell']中的叶子
    ######################################################
    valid_cells = set(sc['cell'])
    leaves_after_first_filter = [term for term in terminals if term.name != 'ref' and term.name in valid_cells]
    
    to_prune_first = [term for term in terminals if term not in leaves_after_first_filter]
    for leaf in to_prune_first:
        tree.prune(leaf)
    ######################################################
    # 第二步：进一步根据organ过滤
    ######################################################
    cell_to_organ = dict(zip(sc['cell'], sc['organ']))
    
    organs_in_leaves = [cell_to_organ[term.name] for term in leaves_after_first_filter]
    from collections import Counter
    organ_freq_in_tree = Counter(organs_in_leaves)
    ######################################################
    df = pd.DataFrame.from_dict(organ_freq_in_tree, orient='index').reset_index()
    df.columns = ['organ', 'count']
    df['freq'] = df['count'] / df['count'].sum()
    df_filtered = df[~(df['count'] < 5)]
    # df_filtered = df
    if len(df_filtered) >=2:
        df_filtered.to_csv(out_dir+"/"+metgroup.index[my_clone]+"_cellNumber.csv", index=False)
        ######################################################
        total_leaves = len(leaves_after_first_filter)
        # organ_freq_in_tree = {k: v / total_leaves for k, v in organ_freq_in_tree.items()}
        organ_freq_in_tree = {k: v for k, v in organ_freq_in_tree.items()}
        
        leaves_after_second_filter = []
        for term in leaves_after_first_filter:
            organ = cell_to_organ[term.name]
            if organ_freq_in_tree.get(organ, 0) >= 5:
                leaves_after_second_filter.append(term)
        
        to_prune_second = [term for term in leaves_after_first_filter if term not in leaves_after_second_filter]
        for leaf in to_prune_second:
            tree.prune(leaf)
        
        Phylo.write(tree, outfile, "newick")
        print(f"Processed clone {my_clone}, saved pruned tree to {outfile}")


# 2. run fitchCount

In [35]:
sc['organOri']=sc['organ']
replace_dict = {
    'LvM-1': 'LvM',
    'LvM-2': 'LvM',
    'LvM-3': 'LvM',
    'LvM-4': 'LvM',
    'LvmM': 'LvM',
    'OvaM-1': 'OvaM',
    'OvaM-2': 'OvaM'
}
# # 执行替换操作
sc['organ'] = sc['organ'].replace(replace_dict)
organOrders = ['PT', 'LuM', 'ThyM', 'LvM', 'OvaM', 'SpnM', 'IntM']
tree_files = glob.glob("*.nwk")
for treefile in tree_files:
    outname = treefile.replace("_pruned.nwk", "")
    with open(treefile, 'r') as f:
        tree = f.read()
    cassiopeia_tree=cas.data.CassiopeiaTree(tree=tree)
    ################################    
    cell_meta = pd.DataFrame(index=cassiopeia_tree.leaves)
    cell_meta["cell"] = cassiopeia_tree.leaves
    cell_meta["dummy"] = "unknown"  
    cassiopeia_tree.cell_meta = cell_meta
    cassiopeia_tree.cell_meta = cassiopeia_tree.cell_meta.merge(sc[['cell', 'organ']], on='cell', how='left')
    # cassiopeia_tree.cell_meta.loc[0, "organ"] = "ref"
    cassiopeia_tree.cell_meta["organ"] = cassiopeia_tree.cell_meta["organ"].astype("category")
    cassiopeia_tree.cell_meta = cassiopeia_tree.cell_meta.set_index("cell")
    mtx=cas.tl.fitch_count(cassiopeia_tree, "organ",infer_ancestral_states=True, state_key='S1', unique_states=None)
    myOrder = [org for org in organOrders if org in mtx.index]
    res = mtx.loc[myOrder,myOrder]
    res.to_csv(outname+"_transitionCount.csv", index=True)
    np.fill_diagonal(res.values,0)
    res = res.apply(lambda x: x / max(1, x.sum()), axis=1)
    # np.fill_diagonal(res.values,0)
    res.to_csv(outname+"_fitchCount.csv", index=True)

In [46]:
out_dir="/syn2/zhaolian/3.JiLab/results/3.PacBio/6.fitchCount_5cells"
os.chdir(out_dir)

files = glob.glob("*_transitionCount.csv")
for myfile in files:
    outname = myfile.replace("_transitionCount.csv", "")
    ################################    
    res = pd.read_csv(myfile,index_col=0)
    res = res.apply(lambda x: x / max(1, x.sum()), axis=1)
    np.fill_diagonal(res.values,0)
    res.to_csv(outname+"_fitchCount_rescale_Diagonal.csv", index=True)